# 📘 智能体架构 10：模拟器 / 心理模型循环

欢迎来到我们系列的第十本笔记本。今天，我们将探索一种专为高风险环境中安全和健壮决策而设计的复杂架构：**模拟器**，也称为**心理模型循环**。

核心思想是让智能体以非常具体的方式"三思而后行"。智能体不会立即在现实世界中执行操作，而是首先在环境的内部模拟版本中测试其拟议的操作。通过在这个安全沙箱中观察可能的后果，它可以评估风险、完善策略，然后才在现实中执行更深思熟虑的操作。

我们将构建一个简单的**股票交易智能体**来演示这一点。"现实世界"将是一个逐步推进的市场模拟器。在进行交易之前，我们的智能体将：
1. 提出一个总体策略（例如，"激进买入"）。
2. 在*分叉的*市场模拟器版本中运行该策略多个未来步骤，以查看潜在结果。
3. 分析模拟结果以评估风险和回报。
4. 做出最终、完善的决定（例如，"模拟显示高波动性；让我们买入较少的金额。"）。
5. 在真实市场中执行该完善的交易。

这一模式对于将智能体从信息任务转移到在现实世界中执行操作至关重要，其中错误可能产生实际后果。

### 定义
**模拟器**或**心理模型循环**架构涉及一个智能体，该智能体使用其环境的内部模型来模拟潜在操作的结果，然后再执行其中任何操作。这允许智能体执行假设分析、预测后果并完善其计划以提高安全性和有效性。

### 高层工作流程

1. **观察：** 智能体观察现实环境的当前状态。
2. **提议操作：** 基于其目标和当前状态，智能体的规划模块生成一个高级拟议操作或策略。
3. **模拟：** 智能体将环境的当前状态分叉到沙盒模拟中。它应用拟议的操作并向前运行模拟以观察一系列可能的结果。
4. **评估和改进：** 智能体分析模拟的结果。操作是否产生了期望的结果？是否有意外的不利后果？基于此评估，它将其初始提议完善为最终的、具体的操作。
5. **执行：** 智能体在*真实*环境中执行最终的、完善的操作。
6. **重复：** 循环从现实环境的新状态重新开始。

### 适用场景 / 应用
* **机器人技术：** 在移动机械臂之前模拟抓取或路径以避免碰撞或损坏。
* **高风险决策：** 在金融领域，在不同市场条件下模拟交易对投资组合的影响。在医疗保健领域，模拟治疗计划的潜在效果。
* **复杂游戏 AI：** 策略游戏中的 AI 模拟多步 ahead 以选择最佳操作。

### 优缺点
* **优点：**
    * **安全和风险降低：** 通过首先在安全环境中审查操作，大幅减少有害或昂贵错误的机会。
    * **提高性能：** 通过允许前瞻和规划，导致更健壮和深思熟虑的决策。
* **缺点：**
    * **模拟-现实差距：** 有效性完全取决于模拟器的保真度。如果世界模型不准确，智能体的计划可能基于错误的假设。
    * **计算成本：** 运行模拟（尤其是多个场景）计算昂贵且比直接行动更慢。

## 阶段 0：基础与环境设置

我们将安装库并设置我们的环境。

In [ ]:
# !pip install -q -U langchain-openai langchain langgraph rich python-dotenv numpy

In [ ]:
import os
import random
import numpy as np
from typing import List, Dict, Any, Optional
from dotenv import load_dotenv

# Pydantic for data modeling
from pydantic import BaseModel, Field

# LangChain components
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# LangGraph components
from langgraph.graph import StateGraph, END
from typing_extensions import TypedDict

# For pretty printing
from rich.console import Console
from rich.markdown import Markdown
from rich.table import Table

# --- API Key and Tracing Setup ---
load_dotenv()

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "Agentic Architecture - Simulator (OpenAI)"

required_vars = ["OPENAI_API_KEY", "LANGCHAIN_API_KEY"]
for var in required_vars:
    if var not in os.environ:
        print(f"Warning: Environment variable {var} not set.")

print("Environment variables loaded and tracing is set up.")

## 阶段 1：构建模拟器环境

首先，我们需要创建我们的智能体将与之交互的"世界"。我们将构建一个 `MarketSimulator` 类，用于管理股票、投资组合的状态，并包括一个 `step` 函数来推进时间。这将作为我们的"现实世界"和智能体模拟的沙盒。

In [ ]:
console = Console()

class Portfolio(BaseModel):
    cash: float = 10000.0
    shares: int = 0
    
    def value(self, current_price: float) -> float:
        return self.cash + self.shares * current_price

class MarketSimulator(BaseModel):
    """A simple simulation of a stock market for one asset."""
    day: int = 0
    price: float = 100.0
    volatility: float = 0.1 # Standard deviation for price changes
    drift: float = 0.01 # General trend
    market_news: str = "Market is stable."
    portfolio: Portfolio = Field(default_factory=Portfolio)

    def step(self, action: str, amount: float = 0.0):
        """Advance the simulation by one day, executing a trade first."""
        # 1. Execute trade
        if action == "buy": # amount is number of shares
            shares_to_buy = int(amount)
            cost = shares_to_buy * self.price
            if self.portfolio.cash >= cost:
                self.portfolio.shares += shares_to_buy
                self.portfolio.cash -= cost
        elif action == "sell": # amount is number of shares
            shares_to_sell = int(amount)
            if self.portfolio.shares >= shares_to_sell:
                self.portfolio.shares -= shares_to_sell
                self.portfolio.cash += shares_to_sell * self.price
        
        # 2. Update market price (Geometric Brownian Motion)
        daily_return = np.random.normal(self.drift, self.volatility)
        self.price *= (1 + daily_return)
        
        # 3. Advance time
        self.day += 1
        
        # 4. Potentially update news
        if random.random() < 0.1: # 10% chance of new news
            self.market_news = random.choice(["Positive earnings report expected.", "New competitor enters the market.", "Macroeconomic outlook is strong.", "Regulatory concerns are growing."])
            # News affects drift
            if "Positive" in self.market_news or "strong" in self.market_news:
                self.drift = 0.05
            else:
                self.drift = -0.05
        else:
             self.drift = 0.01 # Revert to normal drift

    def get_state_string(self) -> str:
        return f"Day {self.day}: Price=${self.price:.2f}, News: {self.market_news}\nPortfolio: ${self.portfolio.value(self.price):.2f} ({self.portfolio.shares} shares, ${self.portfolio.cash:.2f} cash)"

print("Market simulator environment defined successfully.")

Market simulator environment defined successfully.


## 阶段 2：构建模拟器智能体

现在我们将使用 LangGraph 编排 `观察 → 提议 → 模拟 → 改进 → 执行` 工作流程。我们将定义 LLM 输出的 Pydantic 模型，以确保步骤之间的结构化通信。

In [ ]:
llm = ChatOpenAI(model="gpt-4o", temperature=0.4)

# Pydantic models for structured LLM outputs
class ProposedAction(BaseModel):
    """The high-level strategy proposed by the analyst.""" 
    strategy: str = Field(description="A high-level trading strategy, e.g., 'buy aggressively', 'sell cautiously', 'hold'.")
    reasoning: str = Field(description="Brief reasoning for the proposed strategy.")

class FinalDecision(BaseModel):
    """The final, concrete action to be executed."""
    action: str = Field(description="The final action to take: 'buy', 'sell', or 'hold'.")
    amount: float = Field(description="The number of shares to buy or sell. Should be 0 if holding.")
    reasoning: str = Field(description="Final reasoning, referencing the simulation results.")

# LangGraph State
class AgentState(TypedDict):
    real_market: MarketSimulator
    proposed_action: Optional[ProposedAction]
    simulation_results: Optional[List[Dict]]
    final_decision: Optional[FinalDecision]

# Graph Nodes
def propose_action_node(state: AgentState) -> Dict[str, Any]:
    """Observes the market and proposes a high-level strategy."""
    console.print("--- 🧐 分析师提议操作 ---")
    prompt = ChatPromptTemplate.from_template(
        "你是一位敏锐的金融分析师。根据当前的市场状态，提出交易策略。\n\n市场状态：\n{market_state}"
    )
    proposer_llm = llm.with_structured_output(ProposedAction)
    chain = prompt | proposer_llm
    proposal = chain.invoke({"market_state": state['real_market'].get_state_string()})
    console.print(f"[yellow]提议：[/yellow] {proposal.strategy}。[italic]原因：{proposal.reasoning}[/italic]")
    return {"proposed_action": proposal}

def run_simulation_node(state: AgentState) -> Dict[str, Any]:
    """在沙盒模拟中运行拟议的策略。"""
    console.print("--- 🤖 运行模拟 ---")
    strategy = state['proposed_action'].strategy
    num_simulations = 5
    simulation_horizon = 10 # days
    results = []

    for i in range(num_simulations):
        # 重要：创建深拷贝以不影响真实市场状态
        simulated_market = state['real_market'].model_copy(deep=True)
        initial_value = simulated_market.portfolio.value(simulated_market.price)

        # 将策略转换为模拟的具体操作
        if "buy" in strategy:
            action = "buy"
            # 激进 = 现金的 25%，谨慎 = 10%
            amount = (simulated_market.portfolio.cash * (0.25 if "aggressively" in strategy or "激进" in strategy else 0.1)) / simulated_market.price
        elif "sell" in strategy:
            action = "sell"
            # 激进 = 持仓的 25%，谨慎 = 10%
            amount = simulated_market.portfolio.shares * (0.25 if "aggressively" in strategy or "激进" in strategy else 0.1)
        else:
            action = "hold"
            amount = 0
        
        # 向前运行模拟
        simulated_market.step(action, amount)
        for _ in range(simulation_horizon - 1):
            simulated_market.step("hold") # 初始操作后只是持有
        
        final_value = simulated_market.portfolio.value(simulated_market.price)
        results.append({"sim_num": i+1, "initial_value": initial_value, "final_value": final_value, "return_pct": (final_value - initial_value) / initial_value * 100})
    
    console.print("[cyan]模拟完成。结果将传递给风险管理者。[/cyan]")
    return {"simulation_results": results}

def refine_and_decide_node(state: AgentState) -> Dict[str, Any]:
    """分析模拟结果并做出最终、完善的决定。"""
    console.print("--- 🧠 风险管理者完善决策 ---")
    results_summary = "\n".join([f"模拟 {r['sim_num']}：初始=${r['initial_value']:.2f}，最终=${r['final_value']:.2f}，回报={r['return_pct']:.2f}%" for r in state['simulation_results']])
    
    prompt = ChatPromptTemplate.from_template(
        "你是一位谨慎的风险管理者。你的分析师提出了一项策略。你已运行模拟来测试它。根据潜在结果，做出最终、具体的决定。如果结果高度变化或为负，则降低风险（例如，买入/卖出较少的股票，或持有）。\n\n初始提议：{proposal}\n\n模拟结果：\n{results}\n\n真实市场状态：\n{market_state}"
    )
    decider_llm = llm.with_structured_output(FinalDecision)
    chain = prompt | decider_llm
    final_decision = chain.invoke({
        "proposal": state['proposed_action'].strategy,
        "results": results_summary,
        "market_state": state['real_market'].get_state_string()
    })
    console.print(f"[green]最终决定：[/green] {final_decision.action} {final_decision.amount:.0f} 股。[italic]原因：{final_decision.reasoning}[/italic]")
    return {"final_decision": final_decision}

def execute_in_real_world_node(state: AgentState) -> Dict[str, Any]:
    """在真实市场环境中执行最终决定。"""
    console.print("--- 🚀 在现实世界中执行 ---")
    decision = state['final_decision']
    real_market = state['real_market']
    real_market.step(decision.action, decision.amount)
    console.print(f"[bold]执行完成。新市场状态：[/bold]\n{real_market.get_state_string()}")
    return {"real_market": real_market}

# Build the graph
workflow = StateGraph(AgentState)
workflow.add_node("propose", propose_action_node)
workflow.add_node("simulate", run_simulation_node)
workflow.add_node("refine", refine_and_decide_node)
workflow.add_node("execute", execute_in_real_world_node)

workflow.set_entry_point("propose")
workflow.add_edge("propose", "simulate")
workflow.add_edge("simulate", "refine")
workflow.add_edge("refine", "execute")
workflow.add_edge("execute", END)

simulator_agent = workflow.compile()
print("Simulator-in-the-loop agent graph compiled successfully.")

## 阶段 3：演示

让我们在市场中运行我们的智能体几天。我们将从一些好消息开始，看看它如何反应，然后引入一些坏消息。

In [5]:
real_market = MarketSimulator()
console.print("--- Initial Market State ---")
console.print(real_market.get_state_string())

# --- Day 1 Run ---
console.print("\n--- Day 1: Good News Hits! ---")
real_market.market_news = "Positive earnings report expected."
real_market.drift = 0.05
initial_state = {"real_market": real_market}
final_state = simulator_agent.invoke(initial_state)
real_market = final_state['real_market']

# --- Day 2 Run ---
console.print("\n--- Day 2: Bad News Hits! ---")
real_market.market_news = "New competitor enters the market."
real_market.drift = -0.05
initial_state = {"real_market": real_market}
final_state = simulator_agent.invoke(initial_state)
real_market = final_state['real_market']

--- Initial Market State ---


Day 0: Price=$100.00, News: Market is stable.
Portfolio: $10000.00 (0 shares, $10000.00 cash)



---  Day 1: Good News Hits! ---


--- 🧐 Analyst Proposing Action ---
Proposal: buy aggressively. Reason: The positive earnings report is a strong bullish signal, and the market is already stable. This is a good opportunity to enter a position before the price potentially rises further.
--- 🤖 Running Simulations ---
Simulation complete. Results will be passed to the risk manager.
--- 🧠 Risk Manager Refining Decision ---
Final Decision: buy 20 shares. Reason: The simulations confirm a strong upward trend, with all scenarios resulting in a positive return. The analyst's proposal to buy aggressively is validated. I will execute a significant but not excessive purchase of 20 shares to capitalize on the expected price increase while maintaining a cash reserve.
--- 🚀 Executing in Real World ---
Execution complete. New market state:
Day 1: Price=$99.16, News: Market is stable.
Portfolio: $7983.18 (20 shares, $8000.00 cash)



--- Day 2: Bad News Hits! ---


--- 🧐 Analyst Proposing Action ---
Proposal: sell cautiously. Reason: The entry of a new competitor introduces significant uncertainty and potential downside risk. While the price hasn't dropped dramatically yet, it's prudent to reduce exposure.
--- 🤖 Running Simulations ---
Simulation complete. Results will be passed to the risk manager.
--- 🧠 Risk Manager Refining Decision ---
Final Decision: sell 5 shares. Reason: The simulations show a high degree of variance and a negative average return, confirming the analyst's concerns. The initial proposal to sell cautiously is sound. I will de-risk the portfolio by selling 5 shares (25% of the position) to lock in some cash and reduce exposure to the potential downside from the new competitor.
--- 🚀 Executing in Real World ---
Execution complete. New market state:
Day 2: Price=$93.81, News: Market is stable.
Portfolio: $9802.90 (15 shares, $8395.73 cash)


### 结果分析

智能体的行为展示了模拟循环的价值：

- **在第 1 天（好消息）：**
    - *分析师*提出了激进买入，看到了机会。
    - *模拟器*确认了正面结果的极高概率。
    - *风险管理者*将激进策略转化为具体的、大量购买（20 股），但没有冒险整个现金余额。

- **在第 2 天（坏消息）：**
    - *分析师*正确地识别了新风险并提出了谨慎卖出。
    - *模拟器*可能显示了一系列结果，有些场景显示急剧下跌，其他显示恢复，证实了不确定性。
    - *风险管理者*看到模拟中的差异和负平均回报，做出了减少头寸的决定（卖出 5 股），而不是恐慌性抛售全部持仓。这是一个比简单的基于规则的智能体可能采取的行动更细致的行动。

没有模拟循环的天真智能体可能在第 1 天买入太多，然后在第 2 天卖出所有，产生更高的交易成本，并可能错过复苏。我们的模拟器智能体的行为更像真实世界的交易者，做出概率性押注，然后在新信息改变了风险状况时对冲押注。

## 结论

在本笔记本中，我们构建了一个强大的智能体架构，它使用内部**模拟器**在承诺操作之前测试和完善其操作。通过创建 `提议 → 模拟 → 改进 → 执行` 循环，我们使我们的智能体能够执行复杂的风险分析，并在动态环境中做出更细致、更安全的决策。

这一模式是构建可以在现实世界中安全有效地操作的智能体的基石。在内部"心理模型"上执行假设分析的能力使智能体能够预测后果、避免昂贵错误，并制定更健壮的策略。虽然模拟器的保真度是一个关键依赖（"模拟-现实差距"），但这一架构为构建负责任的、采取行动的 AI 提供了一个清晰且可扩展的框架。